# FrogDQ — Full Pipeline Walkthrough

This notebook runs the complete FrogDQ experiment pipeline step by step, **reusing
the existing project code** (`scripts/*.py`, `frogdq/*`) and the shared `config.yaml`
— it mirrors `run_pipeline.sh` one cell at a time:

1. **Select Datasets** — `scripts/select_datasets.py`
2. **Poison Data** — `scripts/poison_data.py`
3–7. **Data Preparation Pipelines** (the data-cleaning benchmarks FrogDQ is compared
   against): Custom Pipeline (CP), Saga++, AutoGluon, Baseline-0, KNN imputation
8. **Main Experiments** — Optuna hyperparameter search via `frogdq.optimization.OptunaExperiment`,
   run with `warm_start=False` so studies live **only in memory** (no `optuna_studies.db`
   is ever written) and results are collected straight into a plain Python `dict`
9. **Results Evaluation** — Friedman test + critical-difference diagrams comparing
   `clean` against every other data-handling method, separately for AR and NAR corruption

> **Heads up:** on the full `config.yaml` (~70 datasets, up to 256 Optuna trials ×
> 5 seeds each, plus several heavy data-preparation benchmarks) this pipeline can
> take a very long time end-to-end. Run cells selectively, or temporarily shrink
> `config['datasets'] / ['n_trials'] / ['n_seeds']` in the cell below if you just
> want to exercise the notebook.


In [1]:
import sys
import copy
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / 'config.yaml'
PYTHON = sys.executable


def run_script(args):
    '''Run `python <script> <args...>` from the repo root — the same way run_pipeline.sh does.'''
    cmd = [PYTHON, *args]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## Configuration

Load the shared `config.yaml` with `frogdq.optimization.load_config` — every step below reads its settings from this single object, exactly like `main.py` and the CLI scripts do.

In [3]:
from frogdq.optimization import load_config

config = load_config(CONFIG_PATH)
print(f"Datasets:     {len(config['datasets'])}")
print(f"Data modes:   {config['data_modes']}")
print(f"Model types:  {config['model_types']}")
print(f"Trials/seeds: {config['n_trials']} trials x {config['n_seeds']} seeds")
print(f"Benchmarks:   autogluon={config.get('run_autogluon')}, cp={config.get('run_cp')}, "
      f"saga={config.get('run_saga')}, baseline_zero={config.get('run_baseline_zero')}, "
      f"knn={config.get('run_knn')}")


Datasets:     5
Data modes:   ['clean', 'ar', 'nar']
Model types:  ['linear', 'mlp']
Trials/seeds: 256 trials x 10 seeds
Benchmarks:   autogluon=False, cp=True, saga=True, baseline_zero=False, knn=False


## Step 1 — Select Datasets

`scripts/select_datasets.py` filters the OpenML dataset catalogue (size filter →
family/redundancy filter → pilot-performance filter, see `step1_size_filter` /
`step2_family_filter` / `step3_pilot_filter` in that script) down to the subset
used by the rest of the pipeline, and — via `--output` — (re)writes the resulting
list directly into the `datasets:` block of `config.yaml`, just like
`run_pipeline.sh` does.

> ⚠️ This **overwrites `config.yaml` in place** (same as the real pipeline). If you
> want to keep the current dataset list, back the file up first or pass a different
> `--output` path below.


In [ ]:
run_script([
    'scripts/select_datasets.py',
    '--data-dir', 'data',
    '--config', str(CONFIG_PATH),
    '--output', str(CONFIG_PATH),
])

# Reload — the script may have rewritten the `datasets:` list in config.yaml
config = load_config(CONFIG_PATH)
print(f"\n{len(config['datasets'])} datasets selected: {config['datasets']}")


## Step 2 — Poison Data

`scripts/poison_data.py` reads the `poisoning:` block of `config.yaml` and, for
every selected dataset, produces an **AR** (At Random) and a **NAR** (Not At
Random) corrupted copy plus a boolean corruption mask under `data_poisoned/`,
together with a held-out clean test split under `data_poisoned/test/`. Every
downstream step (poisoning-aware loading in `frogdq.data.load_data`, the
data-preparation benchmarks, and the Optuna experiments) consumes these files.


In [ ]:
run_script([
    'scripts/poison_data.py',
    '--input_dir', 'data',
    '--output_dir', 'data_poisoned',
    '--config', str(CONFIG_PATH),
])


## Steps 3–7 — Data Preparation Pipelines (benchmarks)

These steps pre-compute the alternative data-cleaning / imputation approaches
that FrogDQ is benchmarked against in Step 8: the **Custom Pipeline (CP)**,
**Saga++**, **AutoGluon**, **Baseline-0** (zero/random imputation) and **KNN**
imputation. Each one is only required if its `run_*` flag is enabled in
`config.yaml` — the cells below check the flag before launching the (often
expensive) script, exactly like `run_pipeline.sh` does.


In [ ]:
# Step 3 — Custom Pipeline (CP): MICE + IQR outlier handling + association-rule repair
if config.get('run_cp', False):
    run_script([
        'scripts/data_preparation_pipeline.py',
        '--input_dir', 'data_poisoned',
        '--output_dir', 'data_cleaned_cp',
        '--config', str(CONFIG_PATH),
    ])
else:
    print("run_cp is false in config.yaml — skipping the CP benchmark.")


In [ ]:
# Step 4 — Saga++: genetic-search data-repair pipeline
if config.get('run_saga', False):
    run_script([
        'scripts/saga.py',
        '--input_dir', 'data_poisoned',
        '--output_dir', 'data_cleaned_saga',
        '--config', str(CONFIG_PATH),
    ])
else:
    print("run_saga is false in config.yaml — skipping the Saga++ benchmark.")


In [ ]:
# Step 5 — AutoGluon: pre-computed AutoGluon-transformed features
if config.get('run_autogluon', False):
    run_script([
        'scripts/autogluon.py',
        '--config', str(CONFIG_PATH),
        '--output-dir', 'data_autogluon',
        '--data-dir', 'data',
        '--poisoned-dir', 'data_poisoned',
    ])
else:
    print("run_autogluon is false in config.yaml — skipping the AutoGluon benchmark.")


In [ ]:
# Step 6 — Baseline 0: numerical NaN -> 0, categorical NaN -> random observed value
if config.get('run_baseline_zero', False):
    run_script([
        'scripts/baseline_zero.py',
        '--input_dir', 'data_poisoned',
        '--output_dir', 'data_baseline_zero',
        '--config', str(CONFIG_PATH),
    ])
else:
    print("run_baseline_zero is false in config.yaml — skipping the Baseline-0 benchmark.")


In [ ]:
# Step 7 — KNN imputation
if config.get('run_knn', False):
    run_script([
        'scripts/knn.py',
        '--input_dir', 'data_poisoned',
        '--output_dir', 'data_knn',
        '--config', str(CONFIG_PATH),
    ])
else:
    print("run_knn is false in config.yaml — skipping the KNN-imputation benchmark.")


## Step 8 — Main Experiments

We instantiate `OptunaExperiment` (see `frogdq/optimization.py`) directly from
`config` and run the full hyperparameter search across every
`(dataset × data_mode × model_type × curriculum/gate × preparation)` combination
via `run_all_experiments()` — the same orchestration `main.py` uses — then collapse
the resulting per-trial / per-seed table into a plain `dict` for the analysis in
Step 9, **without ever touching `optuna_studies.db`**:

- `OptunaExperiment.run_experiment` only opens a persistent SQLite study when
  `warm_start=True`; with `warm_start=False` it creates the study with
  `optuna.create_study(storage=None, ...)` — an **in-memory** study
  (see `frogdq/optimization.py` around `storage=self.storage_url if self.warm_start else None`).
  We therefore force `warm_start=False` on a working copy of the config.
- `run_all_experiments()` still returns the combined results as an in-memory
  `DataFrame` (and writes a couple of CSVs as a side effect, same as the CLI) — we
  use that `DataFrame` only to build our own `results` dictionary below; nothing
  from this step depends on reading anything back from disk.


In [2]:
from frogdq.optimization import OptunaExperiment

experiment_config = copy.deepcopy(config)
experiment_config['warm_start'] = False  # -> optuna.create_study(storage=None, ...): in-memory only, no .db file

# Resolve data directories to absolute paths so the experiment works regardless
# of the notebook's working directory (notebooks/ != repo root).
_dir_keys = ('data_dir', 'poisoned_dir',
             'autogluon_data_dir', 'cp_data_dir', 'saga_data_dir',
             'baseline_zero_data_dir', 'knn_data_dir')
_defaults = ('data', 'data_poisoned',
             'data_autogluon', 'data_cleaned_cp', 'data_cleaned_saga',
             'data_baseline_zero', 'data_knn')
for key, default in zip(_dir_keys, _defaults):
    experiment_config[key] = str(REPO_ROOT / experiment_config.get(key, default))

experiment = OptunaExperiment(experiment_config)
combined_results = experiment.run_all_experiments()
combined_results.head()

NameError: name 'config' is not defined

### Build a per-bootstrap results dictionary

For every `(dataset, data_mode, model_type, curriculum/gate/preparation)`
combination we keep the **best Optuna trial** (`rank == 1`, selected by average
*validation* metric) and record the **test** metric (`test_f1` for classification,
`test_r2` for regression) **separately for each of the `n_seeds` bootstrap runs**,
storing `(dataset, seed)` as the key rather than collapsing to a mean.

Keeping the per-seed scores intact means the Friedman test will use
`n_datasets × n_seeds` blocks instead of just `n_datasets` — preserving the full
variance information from the bootstrap resampling and giving the test more power.
Each `(dataset, seed)` pair is a genuinely independent sample of the joint
(data-split, model-initialisation) randomness (see `frogdq/data.py:316,334`
and `frogdq/training.py:551`).

Method names are derived from `(data_mode, use_curriculum, use_gate, preparation)`:

| condition                              | method            |
|----------------------------------------|-------------------|
| `data_mode == 'clean'`                 | `clean`           |
| `preparation != 'standard'`            | `autogluon` / `cp` / `saga` / `baseline_zero` / `knn` |
| `use_curriculum and use_gate`          | `gate+curriculum` |
| `use_gate` only                        | `gate`            |
| `use_curriculum` only                  | `curriculum`      |
| neither                                | `baseline`        |

The result is `results[model_type][data_mode][method][(dataset, seed)] -> test_metric`.


In [ ]:
def method_label(data_mode, use_curriculum, use_gate, preparation):
    if data_mode == 'clean':
        return 'clean'
    if preparation != 'standard':
        return preparation
    if use_curriculum and use_gate:
        return 'gate+curriculum'
    if use_gate:
        return 'gate'
    if use_curriculum:
        return 'curriculum'
    return 'baseline'


def best_trial_seed_scores(trial_rows):
    '''Per-seed test metric of the best (rank == 1) trial. Returns {seed: metric}.'''
    best = trial_rows[trial_rows['rank'] == trial_rows['rank'].min()]
    metric_col = 'test_f1' if 'test_f1' in best.columns and best['test_f1'].notna().any() else 'test_r2'
    return dict(zip(best['seed'].astype(int), best[metric_col]))


results = {}  # results[model_type][data_mode][method][(dataset, seed)] -> test metric
group_cols = ['model_type', 'data_mode', 'use_curriculum', 'use_gate', 'preparation', 'dataset']
for (model_type, data_mode, use_curriculum, use_gate, preparation, dataset), rows in combined_results.groupby(group_cols):
    method = method_label(data_mode, use_curriculum, use_gate, preparation)
    seed_scores = best_trial_seed_scores(rows)
    method_dict = (results.setdefault(model_type, {})
                          .setdefault(data_mode, {})
                          .setdefault(method, {}))
    for seed, score in seed_scores.items():
        method_dict[(dataset, seed)] = score

for model_type, by_mode in results.items():
    for data_mode, by_method in by_mode.items():
        n_blocks = len(next(iter(by_method.values()))) if by_method else 0
        print(f"{model_type:6s} / {data_mode:5s}: methods={sorted(by_method)}, "
              f"blocks per method={len(next(iter(by_method.values())))}")


## Step 9 — Results Evaluation: Friedman test + critical-difference diagrams

For each model type and each corruption regime (**AR**, **NAR**) we compare
`clean` — the reference, which is necessarily the best method since there is no
corruption left to handle — against every method available for that regime:
FrogDQ's own variants (`baseline`, `curriculum`, `gate`, `gate+curriculum`) plus
whichever precomputed-cleaning benchmarks were enabled (`autogluon`, `cp`,
`saga`, `baseline_zero`, `knn`). **AR and NAR are evaluated separately**, since
they are different corruption regimes and pooling them would confound the
ranking.

Each dataset is one Friedman "block": within it, methods are ranked by their
best-trial test metric (higher = better — Friedman only needs *within-block*
ranks, so mixing classification (`F1`) and regression (`R²`) datasets is fine,
each dataset still ranks its own methods consistently). The Friedman test checks
whether the average ranks differ significantly across methods; the
critical-difference diagram (Nemenyi post-hoc, via `scikit-posthocs`) then shows
which pairwise differences are — and aren't — statistically significant.


In [ ]:
try:
    import scikit_posthocs as sp
except ImportError:
    %pip install -q scikit-posthocs
    import scikit_posthocs as sp

assert hasattr(sp, 'critical_difference_diagram'), (
    "scikit-posthocs >= 0.8 is required for critical_difference_diagram; "
    "run `%pip install -U scikit-posthocs` and restart the kernel."
)


In [ ]:
def score_matrix(results, model_type, corruption_mode):
    '''(dataset, seed) x methods matrix — each (dataset, seed) pair is one Friedman block.

    Merges 'clean' scores (from data_mode='clean') with every method run under
    `corruption_mode`. Only blocks present in *every* method are kept so the
    Friedman design stays balanced.
    '''
    methods_scores = {'clean': results[model_type]['clean']['clean'],
                      **results[model_type][corruption_mode]}
    common_blocks = sorted(set.intersection(*(set(s) for s in methods_scores.values())))
    return pd.DataFrame(
        {method: [scores[blk] for blk in common_blocks]
         for method, scores in methods_scores.items()},
        index=pd.MultiIndex.from_tuples(common_blocks, names=['dataset', 'seed']),
    )


def friedman_and_cd_plot(matrix, title):
    '''Friedman test across methods (blocked by (dataset, seed)) + Nemenyi CD diagram.'''
    stat, p_value = friedmanchisquare(*[matrix[c] for c in matrix.columns])

    # Higher metric = better -> the best method should receive rank 1
    avg_ranks = matrix.rank(axis=1, ascending=False).mean()
    nemenyi = sp.posthoc_nemenyi_friedman(matrix.to_numpy())
    nemenyi.index = nemenyi.columns = matrix.columns

    n_datasets = matrix.index.get_level_values('dataset').nunique()
    n_seeds = matrix.index.get_level_values('seed').nunique()
    print(f"{title}")
    print(f"  blocks={len(matrix)} ({n_datasets} datasets x {n_seeds} seeds), "
          f"methods={matrix.shape[1]}, Friedman chi2={stat:.3f}, p={p_value:.3e}")

    plt.figure(figsize=(9, 0.6 * len(matrix.columns) + 1.5))
    sp.critical_difference_diagram(avg_ranks, nemenyi)
    plt.title(f"{title} — Friedman p = {p_value:.2e}")
    plt.tight_layout()
    plt.show()
    return stat, p_value, avg_ranks


In [ ]:
cd_results = {}
for model_type in results:
    if 'clean' not in results[model_type]:
        continue
    for corruption_mode in ('ar', 'nar'):
        if corruption_mode not in results[model_type]:
            continue
        matrix = score_matrix(results, model_type, corruption_mode)
        if matrix.shape[1] < 3 or len(matrix) < 2:
            print(f"Skipping {model_type}/{corruption_mode}: not enough data.")
            continue

        base_title = f"{model_type.upper()} — {corruption_mode.upper()}"

        # ── Plot 1: all methods including clean (upper-bound reference) ──────
        stat, p, avg_ranks = friedman_and_cd_plot(
            matrix,
            title=f"{base_title} | with clean reference",
        )
        cd_results[(model_type, corruption_mode, 'with_clean')] = (stat, p, avg_ranks)

        # ── Plot 2: competitors only — clean removed ──────────────────────────
        # Dropping clean spreads the remaining methods across the full rank range
        # (1 … k-1), maximising sensitivity to differences among them.
        competitors = matrix.drop(columns='clean')
        if competitors.shape[1] < 2:
            print(f"Skipping competitors-only plot for {base_title}: only one method left.")
            continue
        stat2, p2, avg_ranks2 = friedman_and_cd_plot(
            competitors,
            title=f"{base_title} | competitors only (clean excluded)",
        )
        cd_results[(model_type, corruption_mode, 'competitors')] = (stat2, p2, avg_ranks2)
